In [5]:
%reload_ext autoreload

In [34]:
## Load calibration traces
import pandas as pd
from utilities import load_pickle
import pickle

calibration_file_path = "remove_partial/calibration_set.csv"
calibration_prop_path = "remove_partial/calibration_prop.pickle"
test_file_path = "remove_partial/test_set.csv"
test_prop_path = "remove_partial/test_prop.pickle"
prop_index = 0

In [2]:
## Load test traces
test_file_path = "remove_partial/test_set.csv"


In [3]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm

# Impostazione del device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")

Device in use: cpu


In [4]:

# Funzione per caricare ed elaborare i dati dal CSV
def load_trace_data(csv_file, prop_file, prefix_length=2):
    """
    Legge il CSV, ordina per caso e tempo, e costruisce sequenze (prefissi)
    assegnando una proprietà: 1 se la differenza tra A_Submitted e O_Sent (se presente)
    è ≤ 48 ore, altrimenti 0.
    """
    df = pd.read_csv(csv_file)
    property_dict = load_pickle(prop_file)
    df['start_time'] = pd.to_datetime(df['start_time'])
    df = df.sort_values(['case:concept:name', 'start_time'])

    traces, labels = [], []
    short_traces = []
    activity_to_idx = {}
    idx_counter = 1

    for case_id, group in tqdm(df.groupby('case:concept:name'), desc="Extracting prefixes"):
        submitted_time = None
        sent_time = None
        seq = []
        prev_time = None

        for _, row in group.iterrows():
            event_name = row['concept:name']
            timestamp = row['start_time']

            if event_name not in activity_to_idx:
                activity_to_idx[event_name] = idx_counter
                idx_counter += 1

            delta = 0 if prev_time is None else (timestamp - prev_time).total_seconds()
            prev_time = timestamp
            seq.append([activity_to_idx[event_name], delta])


        if len(seq) > prefix_length:
            prop_value = property_dict[case_id]
            short_traces.append(torch.tensor(seq[:prefix_length], dtype=torch.float32))
            traces.append(torch.tensor(seq[:], dtype=torch.float32))
            labels.append(prop_value)

    short_tensor = pad_sequence(short_traces, batch_first=True, padding_value=0)
    traces_tensor = pad_sequence(traces, batch_first=True, padding_value=0)
    labels_tensor = torch.tensor(labels, dtype=torch.float32)
    return short_tensor, traces_tensor, labels_tensor, activity_to_idx

In [5]:
# Funzione wrapper per ottenere le predizioni dal modello
def model_fnc(x, model):
    """
    Riceve in input un array NumPy di forma (n_samples, seq_len, 2), lo converte in tensore,
    esegue la forward pass del modello e restituisce un array NumPy con le probabilità per ciascuna classe.
    La prima colonna è 1-p (classe negativa) e la seconda p (classe positiva).
    """
    model.eval()
    with torch.no_grad():
        x_tensor = torch.tensor(x, dtype=torch.float32).to(device)
        preds = model(x_tensor)
    preds = preds.cpu().numpy().reshape(-1, 1)
    preds_2d = np.hstack([1 - preds, preds])
    return preds_2d


In [6]:
# Funzione wrapper per ottenere le predizioni dal modello
def quantitative_model_fnc(x, qmodel):
    """
    Riceve in input un array NumPy di forma (n_samples, seq_len, 2), lo converte in tensore,
    esegue la forward pass del modello e restituisce un array NumPy con le probabilità per ciascuna classe.
    La prima colonna è 1-p (classe negativa) e la seconda p (classe positiva).
    """
    model.eval()
    with torch.no_grad():
        x_tensor = torch.tensor(x, dtype=torch.float32).to(device)
        rob_preds = qmodel(x_tensor)

    return rob_preds.cpu().numpy()


In [8]:
# Definizione dell'architettura del modello
class AttentionBol(nn.Module):
    def __init__(self, hidden_dim):
        super(AttentionBol, self).__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, lstm_out, mask):
        scores = self.attn(lstm_out).squeeze(-1)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = torch.softmax(scores, dim=1).unsqueeze(1)
        weighted_sum = torch.bmm(attn_weights, lstm_out).squeeze(1)
        return weighted_sum

class TraceClassifierBol(nn.Module):
    def __init__(self, vocab_size, embed_dim, lstm_hidden):
        super(TraceClassifierBol, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.time_linear = nn.Linear(1, 8)
        self.lstm = nn.LSTM(embed_dim + 8, lstm_hidden, batch_first=True, bidirectional=True)
        self.attention = AttentionBol(lstm_hidden * 2)
        self.fc = nn.Linear(lstm_hidden * 2, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        act = x[:, :, 0].long()
        time = x[:, :, 1].unsqueeze(-1)
        emb = self.embedding(act)
        time_feat = self.time_linear(time)
        combined = torch.cat([emb, time_feat], dim=2)
        lstm_out, _ = self.lstm(combined)
        mask = (act != 0).float()
        attn_out = self.attention(lstm_out, mask)
        out = self.fc(attn_out)
        return self.sigmoid(out).squeeze()

# Caricamento dei pesi del modello già allenato
state_dict = torch.load('remove_partial/trace_classifier_bol_prop_0.pt', map_location=device)
vocab_size = state_dict['embedding.weight'].size(0)
model = TraceClassifierBol(vocab_size, embed_dim=16, lstm_hidden=64)
model.to(device)
model.load_state_dict(state_dict)
model.eval()
print("Modello caricato correttamente.")


Modello caricato correttamente.


/tmp/ipykernel_4671/523753578.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('remove_partial/trace_classifier_bol_prop_0.pt', map_location=devi

In [19]:
L = [10,20]#,30]#,40]

In [20]:
# Caricamento del set di calibrazione
Xc = []
Yc = []

for h in range(len(L)):
    print(h)
    calib_short_traces, calib_traces, calib_labels, _ = load_trace_data(calibration_file_path,calibration_prop_path, prefix_length=L[h])
    Xc.append(calib_short_traces.numpy())
    Yc.append(calib_labels.numpy())

0


Extracting prefixes: 100%|████████████████| 4108/4108 [00:01<00:00, 2519.21it/s]


1


Extracting prefixes: 100%|████████████████| 4108/4108 [00:01<00:00, 2643.78it/s]


In [23]:
Yc[0]

array([[ 8.2531344e+04,  2.6372488e+05,  2.2593689e+04],
       [-1.2822535e+04,  2.5882406e+05, -1.8152986e+05],
       [-6.3303871e+04, -1.8742599e+06,            inf],
       ...,
       [ 7.8107898e+04, -1.8720749e+06,            inf],
       [-4.9161353e+05, -3.5520208e+06,  7.5400000e+00],
       [ 6.5099863e+04,  2.0755520e+05,  5.8410000e+01]], dtype=float32)

In [28]:
# Ottenimento delle predizioni sul set di calibrazione
Pc = []
for h in range(len(L)):
  pred_calib = model_fnc(Xc[h], model)
  Pc.append(pred_calib)

  pred_calib_labels = pred_calib.argmax(axis=1)
  A = np.abs(Yc[h][:,prop_index] - pred_calib_labels)
  nerr = len(A[A!=0])
  print(f'Accuracy calib prefix={L[h]}: ', 1- nerr/len(A))


Accuracy calib prefix=10:  0.0
Accuracy calib prefix=20:  0.0


In [35]:
# Caricamento del set di calibrazione
Xt = []
Pt = []
for h in range(len(L)):
  test_short_traces, test_traces, test_labels, _ = load_trace_data(test_file_path, test_prop_path, prefix_length=L[h])
  xt = test_short_traces.numpy()
  Xt.append(xt)
  Pt.append(model_fnc(xt, model))


Extracting prefixes:   0%|                             | 0/3821 [00:00<?, ?it/s]


KeyError: 'Application_1000334415'

In [ ]:
import numpy as np
from numpy.random import rand
import scipy.special
import scipy.spatial
import copy


In [ ]:
from torch.autograd import Variable
cuda = True if torch.cuda.is_available() else False
FloatTensor = torch.cuda.FloatTensor if cuda else torch.FloatTensor
LongTensor = torch.cuda.LongTensor if cuda else torch.LongTensor
cuda

False

# CP Classification
to calibrate predictions over the MiTL Boolean satisfaction

In [ ]:
class ICP_Classification():
	'''
	Inductive Conformal Prediction for a generic binary classification problem whose output is the probability of assigning
	an input point to class 1

	Xc: input points of the calibration set
	Yc: labels corresponding to points in the calibration set
	mondrian_flag: if True computes class conditional p-values
	trained_model: function that takes x as input and returns the prob. of associating it to the positive class

	Remark: the default labels are 0 (negative class) and 1 (positive class)
	Careful: if different labels are considered, used the method set_labels
			(the non conformity scores are not well-defined otherwise)
	'''



	def __init__(self, Xc, Yc, trained_model, mondrian_flag):
		self.Xc = Xc
		self.Yc = Yc
		self.pos_label = 1
		self.neg_label = 0
		self.mondrian_flag = mondrian_flag
		self.trained_model = trained_model
		self.cal_pred_lkh = trained_model(Xc)
		self.calibr_scores = self.get_nonconformity_scores(Yc,self.cal_pred_lkh) # nonconformity scores on the calibration set
		self.q = len(Yc) # number of points in the calibration set


	def set_labels(self, new_pos_label, new_neg_label):
		# Set the labels used in Y
		self.pos_label = new_pos_label
		self.neg_label = new_neg_label


	def get_nonconformity_scores(self, y, pred_lkh, sorting = True):

		if (self.pos_label != 1) or (self.neg_label != 0):
			y[(y==self.pos_label)] = 1
			y[(y==self.neg_label)] = 0

		pred_probs = scipy.special.softmax(pred_lkh, axis=1)
		n_points = len(y)
		ncm = np.array([np.abs(1-pred_probs[i,int(y[i])]) for i in range(n_points)])
		if sorting:
			ncm = np.sort(ncm)[::-1] # descending order
		return ncm


	def get_p_values(self, x):
		'''
		calibr_scores: non conformity measures computed on the calibration set and sorted in descending order
		x: new input points (shape: (n_points,x_dim)

		return: positive p-values, negative p-values

		'''
		pred_lkh = self.trained_model(x) # prob of going to pos class on x
		if self.mondrian_flag:
			alphas_pos = self.calibr_scores[(self.Yc == self.pos_label)]
			alphas_neg = self.calibr_scores[(self.Yc == self.neg_label)]
			q_pos = alphas_pos.shape[0]
			q_neg = alphas_neg.shape[0]
		else:
			alphas_pos = self.calibr_scores
			alphas_neg = self.calibr_scores
			q_pos = self.q
			q_neg = self.q
		n_points = len(pred_lkh)

		A_pos = self.get_nonconformity_scores(self.pos_label*np.ones(n_points), pred_lkh, sorting = False) # calibr scores for positive class
		A_neg = self.get_nonconformity_scores(self.neg_label*np.ones(n_points), pred_lkh, sorting = False) # negative scores for positive class

		p_pos = np.zeros(n_points) # p-value for class 1
		p_neg = np.zeros(n_points) # p-value for class 0
		for k in range(n_points):
			c_pos_a = 0
			c_pos_b = 0
			c_neg_a = 0
			c_neg_b = 0
			for count_pos in range(q_pos):
				if alphas_pos[count_pos] > A_pos[k]:
					c_pos_a += 1
				elif alphas_pos[count_pos] == A_pos[k]:
					c_pos_b += 1
				else:
					break
			for count_neg in range(q_neg):
				if alphas_neg[count_neg] > A_neg[k]:
					c_neg_a += 1
				elif alphas_neg[count_neg] == A_neg[k]:
					c_neg_b += 1
				else:
					break
			p_pos[k] = ( c_pos_a + rand() * (c_pos_b + 1) ) / (q_pos + 1)
			p_neg[k] = ( c_neg_a + rand() * (c_neg_b + 1) ) / (q_neg + 1)
		return p_pos, p_neg


	def get_confidence_credibility(self, p_pos, p_neg):
		# INPUTS: p_pos and p_neg are the outputs returned by the function get_p_values
		# OUTPUT: array containing confidence and credibility [shape: (n_points,2)]
		# 		first column: confidence (1-smallest p-value)
		# 		second column: credibility (largest p-value)
		n_points = len(p_pos)
		confidence_credibility = np.zeros((n_points,2))
		for i in range(n_points):
			if p_pos[i] > p_neg[i]:
				confidence_credibility[i,0] = 1-p_neg[i]
				confidence_credibility[i,1] = p_pos[i]
			else:
				confidence_credibility[i,0] = 1-p_pos[i]
				confidence_credibility[i,1] = p_neg[i]
		return confidence_credibility

	def compute_confidence_credibility(self, x):
		p_pos, p_neg = self.get_p_values(x)

		return self.get_confidence_credibility(p_pos, p_neg)


	def get_prediction_region(self, epsilon, p_pos, p_neg):
		# INPUTS: p_pos and p_neg are the outputs returned by the function get_p_values
		#		epsilon = confidence_level
		# OUTPUT: one-hot encoding of the prediction region [shape: (n_points,2)]
		# 		first column: negative class
		# 		second column: positive class
		n_points = len(p_pos)

		pred_region = np.zeros((n_points,2))
		for i in range(n_points):
			if p_pos[i] > epsilon:
				pred_region[i,1] = 1
			if p_neg[i] > epsilon:
				pred_region[i,0] = 1

		return pred_region

	def get_coverage(self, pred_region, labels):

		n_points = len(labels)

		c = 0
		for i in range(n_points):
			if pred_region[i,int(labels[i])] == 1:
				c += 1

		coverage = c/n_points

		return coverage

	def compute_coverage(self, eps, inputs, outputs):
		p1, p0 = self.get_p_values(x = inputs)

		self.pred_region = self.get_prediction_region(epsilon = eps, p_pos = p1, p_neg = p0)

		return self.get_coverage(self.pred_region, outputs)


	def compute_efficiency(self):

		n_singletons = 0
		n_points = self.pred_region.shape[0]
		for i in range(n_points):
			if np.sum(self.pred_region[i]) == 1:
				n_singletons += 1

		return n_singletons/n_points

In [ ]:
net_fnc = lambda inp: model_fnc(inp, model)
eps = 0.05

CPs = []
PRs = []
for h in range(len(L)):
  cp = ICP_Classification(Xc = Xc[h], Yc = Yc[h], trained_model = net_fnc, mondrian_flag = False)
  CPs.append(cp)

  print("Computing p-values...")
  p1, p0 = cp.get_p_values(x = Xt[h])


  pred_region = cp.get_prediction_region(epsilon = eps, p_pos = p1, p_neg = p0)
  PRs.append(pred_region)
  print("prediction region for epsilon =", eps, ": ", pred_region)
  print("real labels: ", test_labels)
  coverage = cp.get_coverage(pred_region, test_labels)

  print("coverage for sign = ", 1-eps, ": ", coverage)



Computing p-values...
prediction region for epsilon = 0.05 :  [[1. 1.]
 [1. 0.]
 [1. 1.]
 ...
 [1. 1.]
 [1. 0.]
 [1. 1.]]
real labels:  tensor([1., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1., 0., 1., 1., 1., 0., 0.,
        0., 1., 0., 1., 1., 0., 1., 1., 1., 0., 1., 1., 0., 0., 0., 1., 0., 0.,
        1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 1., 0., 0., 1., 1., 1., 0., 1., 0., 1., 0., 0., 0., 1., 1., 1., 0.,
        1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 1., 0., 0.,
        0.])
coverage for sign =  0.95 :  0.8021978021978022
Computing p-values...
prediction region for epsilon = 0.05 :  [[1. 1.]
 [1. 0.]
 [1. 1.]
 ...
 [1. 1.]
 [0. 1.]
 [1. 1.]]
real labels:  tensor([1., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1., 0., 1., 1., 1., 0., 0.,
        0., 1., 0., 1., 1., 0., 1., 1., 1., 0., 1., 1., 0., 0., 0., 1., 0., 0.,
        1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 1., 0., 0., 1.,

Computing p-values...
pvalue class 1:  [2.52740095e-01 1.40437554e-04 3.37984073e-01 ... 4.85930261e-01
 9.46326090e-05 3.03355610e-01]
pvalue class 0:  [0.14023316 0.97791109 0.10290245 ... 0.03484047 0.73371539 0.1182094 ]
confidence and credibility:  [[0.85976684 0.2527401 ]
 [0.99985956 0.97791109]
 [0.89709755 0.33798407]
 ...
 [0.96515953 0.48593026]
 [0.99990537 0.73371539]
 [0.8817906  0.30335561]]
prediction region for epsilon = 0.05 :  [[1. 1.]
 [1. 0.]
 [1. 1.]
 ...
 [0. 1.]
 [1. 0.]
 [1. 1.]]
real labels:  tensor([1., 0., 1.,  ..., 0., 0., 1.])
coverage for sign =  0.95 :  0.9706883014917561


# CP Regression
to calibrate predictions over the MiTL time robustness

In [ ]:
class ICP_Regression():
	'''
	Inductive Conformal Prediction for a generic binary classification problem whose output is the probability of assigning
	an input point to class 1

	Xc: input points of the calibration set
	Yc: labels corresponding to points in the calibration set
	mondrian_flag: if True computes class conditional p-values
	trained_model: function that takes x as input and returns the prob. of associating it to the positive class

	Remark: the default labels are 0 (negative class) and 1 (positive class)
	Careful: if different labels are considered, used the method set_labels
			(the non conformity scores are not well-defined otherwise)
	'''



	def __init__(self, Xc, Yc, trained_model):
		self.Xc = Xc
		self.Yc = Yc
		self.output_dim = Yc.shape[1]
		self.trained_model = trained_model
		self.calibr_pred = trained_model(Xc)
		self.q = len(Yc) # number of points in the calibration set



	def get_nonconformity_scores(self, y, y_pred, sorting = True):

		n = len(y)
		ncm = np.empty(n)
		for i in range(n):
			ncm[i] = np.linalg.norm(y[i]-y_pred[i])

		if sorting:
			ncm = np.sort(ncm)[::-1] # descending order
		return ncm


	def get_alpha_threshold(self, eps):

		self.calibr_scores = self.get_nonconformity_scores(self.Yc,self.calibr_pred) # nonconformity scores on the calibration set

		q = 1-eps
		threshold = np.quantile(self.calibr_scores, q)

		return threshold



	def get_coverage(self, epsilon, x_test, y_test):


		y_test_pred = self.trained_model(x_test)
		test_scores = self.get_nonconformity_scores(y_test, y_test_pred)
		n_points = len(y_test)

		self.tau = self.get_alpha_threshold(epsilon)

		c = 0
		for i in range(n_points):
			if test_scores[i] < self.tau:
				c += 1
		coverage = c/n_points

		return coverage


	def get_efficiency(self):

		eff = 2*self.tau # width of the 1d tube

		return eff


In [ ]:
net_fnc = lambda inp: quantitative_model_fnc(inp,qmodel)

cp = ICP_Regression(Xc = X_calib, Yc = cal_robs, trained_model = net_fnc)

eps = 0.05

coverage = cp.get_coverage(eps, X_test, test_robs)
print("1D-Coverage for significance = ", 1-eps, ": ", coverage)